# 03 · Bayesian Beta-Binomial — `foe.bayesian`

**Hexkit surfaces:** `pages/sequential_analysis.py` (revenue-impact overlay),
`pages/experiment_analysis.py` (`get_lift_prior`), `utility/automation_engine.py`.

FOE estimates everything by Monte Carlo: Beta(1 + x, 1 + n − x) posteriors, re-weighted by a
log-normal **lift prior** on `log(p_B / p_A)`. The R side computes the same quantities
**deterministically** by nested numerical integration (`integrate`) over the two posteriors,
with the lift-prior density as a weight. Differences are therefore pure Monte Carlo error,
and the tolerances are set at a few Monte Carlo standard errors.

| FOE function | R reference |
|---|---|
| `run_probability_analysis` — P(B > A), E[(B − A)⁺], E[(A − B)⁺], with and without a skeptical lift prior | nested `integrate` over `dbeta × dbeta × dnorm(log b − log a)` |
| `run_probability_analysis` — P(best) for 3 arms | `integrate(dbeta_i(x) · Π pbeta_j(x))` |
| `run_monetary_projection` — expected uplift / risk | exact integral: `period · AOV · E_w[(b·n_B − a·n_A)⁺] / runtime` (AOV noise is mean-preserving) |
| `get_lift_prior` | `log1p` |

In [1]:
import os
os.environ.setdefault("RPY2_CFFI_MODE", "ABI")  # use rpy2 ABI mode; avoids a noisy API-mode probe
%load_ext rpy2.ipython

import numpy as np
import pandas as pd

from foe.bayesian.operations import BayesianEngine, get_lift_prior
from foe.core.models import BusinessCaseInput, ExperimentInput
from validation_helpers import Checks

checks = Checks()
N_SAMPLES = 1_000_000

## R helper: weighted expectations by nested integration

In [2]:
%%R
# E_w[h(a, b)] over the region lo(a) < b < hi(a), where a ~ Beta(pa), b ~ Beta(pb) and the
# weight is the lift-prior density dnorm(log b - log a; m, s). Unnormalised; divide by the
# same integral with h = 1 over the full range.
nested <- function(pa, pb, m, s, h = function(a, b) 1,
                   lo = function(a) -Inf, hi = function(a) Inf) {
  ra <- qbeta(c(1e-10, 1 - 1e-10), pa[1], pa[2])
  rb <- qbeta(c(1e-10, 1 - 1e-10), pb[1], pb[2])
  inner <- function(a) {
    l <- max(lo(a), rb[1]); u <- min(hi(a), rb[2])
    if (l >= u) return(0)
    integrate(function(b) h(a, b) * dbeta(b, pb[1], pb[2]) * dnorm(log(b) - log(a), m, s),
              l, u, rel.tol = 1e-10)$value
  }
  integrate(function(a) dbeta(a, pa[1], pa[2]) * sapply(a, inner),
            ra[1], ra[2], rel.tol = 1e-8)$value
}

# P(B > A), E[(B-A)+], E[(A-B)+] under the lift prior, for one control/challenger pair.
pair_summary <- function(xa, na, xb, nb, m, s) {
  pa <- c(1 + xa, 1 + na - xa); pb <- c(1 + xb, 1 + nb - xb)
  z <- nested(pa, pb, m, s)
  data.frame(
    prob_beat_control = nested(pa, pb, m, s, lo = function(a) a) / z,
    expected_uplift   = nested(pa, pb, m, s, h = function(a, b) b - a, lo = function(a) a) / z,
    expected_loss     = nested(pa, pb, m, s, h = function(a, b) a - b, hi = function(a) a) / z
  )
}

## 1. Two-arm probabilities, uninformative and skeptical lift priors

In [3]:
cases = [
    # name, control n, control x, challenger n, challenger x, expected lift %, skepticism
    ("small positive, uninformative", 10_000, 300, 10_000, 335,  0.0, "uninformative"),
    ("small positive, skeptical",     10_000, 300, 10_000, 335,  0.0, "skeptical"),
    ("large lift, moderate +5%",       4_000, 120,  4_000, 170,  5.0, "moderate"),
    ("negative, skeptical",           20_000, 900, 20_000, 840,  0.0, "skeptical"),
]
rows = []
for name, na, xa, nb, xb, lift, skep in cases:
    prior = get_lift_prior(lift, skep)
    r = BayesianEngine(seed=1).run_probability_analysis(
        ExperimentInput(visitors=[na, nb], conversions=[xa, xb]),
        lift_prior=prior, n_samples=N_SAMPLES,
    )[0]
    rows.append(dict(name=name, na=na, xa=xa, nb=nb, xb=xb,
                     m=prior.mean_log_lift, s=prior.std_log_lift, lift=lift,
                     prob_beat_control=r.prob_beat_control,
                     expected_uplift=r.expected_uplift, expected_loss=r.expected_loss))
py = pd.DataFrame(rows)
py

,name,na,xa,nb,xb,m,s,lift,prob_beat_control,expected_uplift,expected_loss
0,"small positive, uninformative",10000,300,10000,335,0.00000,1.00,0.0,0.920146,0.003567,0.000090
1,"small positive, skeptical",10000,300,10000,335,0.00000,0.10,0.0,0.866922,0.002305,0.000132
2,"large lift, moderate +5%",4000,120,4000,170,0.04879,0.25,5.0,0.997559,0.010612,0.000003
3,"negative, skeptical",20000,900,20000,840,0.00000,0.10,0.0,0.091657,0.000079,0.002539


In [4]:
%%R -i py -o r_out
r_out <- do.call(rbind, lapply(seq_len(nrow(py)), function(i) {
  s <- py[i, ]
  pair_summary(s$xa, s$na, s$xb, s$nb, s$m, s$s)
}))
r_out$m_check <- log1p(py$lift / 100)
r_out

 prob_beat_control

 expected_uplift

 expected_loss

    m_check


1

        0.92018696

... (14 outputs left out)

        0.09154818

    7.868594e-05

  2.537899e-03

 0.00000000

In [5]:
for i, row in py.iterrows():
    r = r_out.iloc[i]
    checks.add(f"P(B>A)      {row['name']}", row.prob_beat_control, r.prob_beat_control, atol=3e-3)
    checks.add(f"E[(B-A)+]   {row['name']}", row.expected_uplift, r.expected_uplift, atol=1e-6, rtol=0.02)
    checks.add(f"E[(A-B)+]   {row['name']}", row.expected_loss, r.expected_loss, atol=1e-6, rtol=0.02)
    checks.add(f"lift prior mean {row['name']}", row.m, r.m_check, rtol=1e-12)

## 2. Probability of being best, 3 arms

With the default ("uninformative", σ = 1 on the log-lift) prior the importance weights are
almost flat, so the exact unweighted integral is the reference.

In [6]:
n3, x3 = [15_000, 15_000, 15_000], [450, 495, 480]
res3 = BayesianEngine(seed=2).run_probability_analysis(
    ExperimentInput(visitors=n3, conversions=x3), n_samples=N_SAMPLES)
best_py = [r.prob_being_best for r in res3]  # challengers 1 and 2
best_py

[0.6672380622851399, 0.2989841250716492]

In [7]:
%%R -i n3 -i x3 -o best_r
n <- unlist(n3); x <- unlist(x3)
a <- 1 + x; b <- 1 + n - x
p_best <- sapply(seq_along(n), function(i) {
  integrate(function(t) {
    out <- dbeta(t, a[i], b[i])
    for (j in setdiff(seq_along(n), i)) out <- out * pbeta(t, a[j], b[j])
    out
  }, qbeta(1e-10, a[i], b[i]), qbeta(1 - 1e-10, a[i], b[i]), rel.tol = 1e-10)$value
})
best_r <- p_best
best_r

[1]

 0.03349094

 0.66665131

 0.29985775

In [8]:
for i in (1, 2):
    checks.add(f"P(best) arm {i} of 3", best_py[i - 1], best_r[i], atol=3e-3)

## 3. Monetary projection (`run_monetary_projection`)

AOV is drawn log-normally with E[AOV] = AOV and independently of the conversion rates, so

`expected_uplift = projection_period · AOV_B · E_w[(b·n_B − a·n_A)⁺] / runtime_days`

and symmetrically for risk with AOV_A. That is computed exactly in R.

In [9]:
na, xa, nb, xb = 9_000, 270, 9_100, 300
runtime, period = 14, 183
aovs = {"Control": 64.0, "Variant": 66.5}
prior = get_lift_prior(0.0, "moderate")
eng = BayesianEngine(seed=3)
mon = eng.run_monetary_projection(
    visitors=[na, nb], conversions=[xa, xb],
    biz_case=BusinessCaseInput(aovs=aovs, runtime_days=runtime, projection_period=period),
    prob_best_overall=[0.0, 0.0], variant_labels=["Control", "Variant"],
    lift_prior=prior, n_simulations=N_SAMPLES,
)[0]
m, s = prior.mean_log_lift, prior.std_log_lift
aov_a, aov_b = aovs["Control"], aovs["Variant"]
{k: mon[k] for k in ["prob_beat_control", "expected_uplift", "expected_risk"]}

{'prob_beat_control': 0.8898204858322765,
 'expected_uplift': 24849.65,
 'expected_risk': 993.9}

In [10]:
%%R -i na -i xa -i nb -i xb -i runtime -i period -i m -i s -i aov_a -i aov_b -o mon_r
pa <- c(1 + xa, 1 + na - xa); pb <- c(1 + xb, 1 + nb - xb)
z <- nested(pa, pb, m, s)
ratio <- na / nb                       # b * nb > a * na  <=>  b > a * ratio
gain  <- nested(pa, pb, m, s, h = function(a, b) b * nb - a * na, lo = function(a) a * ratio) / z
loss  <- nested(pa, pb, m, s, h = function(a, b) a * na - b * nb, hi = function(a) a * ratio) / z
mon_r <- data.frame(
  prob_beat_control = nested(pa, pb, m, s, lo = function(a) a * ratio) / z,
  expected_uplift   = period * aov_b * gain / runtime,
  expected_risk     = period * aov_a * loss / runtime
)
mon_r

 prob_beat_control

 expected_uplift

 expected_risk


1

         0.8898065

        24820.72

      995.2762

In [11]:
checks.add("monetary P(B>A)", mon["prob_beat_control"], mon_r.prob_beat_control.iloc[0], atol=3e-3)
checks.add("monetary expected uplift", mon["expected_uplift"], mon_r.expected_uplift.iloc[0], rtol=0.02)
checks.add("monetary expected risk", mon["expected_risk"], mon_r.expected_risk.iloc[0], rtol=0.03)

True

## Summary

In [12]:
checks.report()

21 checks, 21 passed, 0 failed

Largest differences:
                               check   hexkit        R    abs_diff             tolerance
            monetary expected uplift  24849.7  24820.7      28.934 atol=1e-08, rtol=0.02
              monetary expected risk    993.9  995.276     1.37618 atol=1e-08, rtol=0.03
                  P(best) arm 2 of 3 0.298984 0.299858 0.000873622    atol=0.003, rtol=0
                  P(best) arm 1 of 3 0.667238 0.666651 0.000586748    atol=0.003, rtol=0
P(B>A)      large lift, moderate +5% 0.997559 0.997417 0.000141646    atol=0.003, rtol=0


,check,hexkit,R,abs_diff,tolerance,pass
0,"P(B>A) small positive, uninformative",0.920146,0.920187,4.119055e-05,"atol=0.003, rtol=0",True
1,"E[(B-A)+] small positive, uninformative",0.003567,0.003568,9.455795e-07,"atol=1e-06, rtol=0.02",True
2,"E[(A-B)+] small positive, uninformative",0.000090,0.000090,3.183132e-07,"atol=1e-06, rtol=0.02",True
3,"lift prior mean small positive, uninformative",0.000000,0.000000,0.000000e+00,"atol=1e-08, rtol=1e-12",True
4,"P(B>A) small positive, skeptical",0.866922,0.866881,4.090731e-05,"atol=0.003, rtol=0",True
5,"E[(B-A)+] small positive, skeptical",0.002305,0.002306,1.531863e-06,"atol=1e-06, rtol=0.02",True
6,"E[(A-B)+] small positive, skeptical",0.000132,0.000131,3.100307e-07,"atol=1e-06, rtol=0.02",True
7,"lift prior mean small positive, skeptical",0.000000,0.000000,0.000000e+00,"atol=1e-08, rtol=1e-12",True
8,"P(B>A) large lift, moderate +5%",0.997559,0.997417,1.416460e-04,"atol=0.003, rtol=0",True
9,"E[(B-A)+] large lift, moderate +5%",0.010612,0.010610,1.315915e-06,"atol=1e-06, rtol=0.02",True
